# ERA LTE × climate-toolkit — Colab walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/era_lte_colab.ipynb)

Use climate-toolkit 0.2.0 with ERA long-term experiment (LTE) records to explore how maize yields at Gourton Farm vary with seasonal rainfall. The exercise uses a registered Earth Engine project and compares reported treatment and control yields under low- and high-rainfall conditions.

## 1. Set up

In [ ]:
import sys
assert sys.version_info >= (3, 12), "Use a Python 3.12+ runtime for direct cloud datasets."
%pip install -q "climate-toolkit[cloud]==0.2.0"


## 2. Check the toolkit's core modules

In [ ]:
import climate_toolkit as ct

print(f"climate_toolkit v{ct.__version__}")
print("Public API:", [n for n in ct.__all__ if not n.startswith("__")])
assert ct.__version__ == "0.2.0", "Restart the runtime after installation."


## 3. Connect to Earth Engine

In [ ]:
# Use your own registered Earth Engine project, not another instructor's project.
import os
GCP_PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "")
if not GCP_PROJECT_ID:
    try:
        from google.colab import userdata
        GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
    except Exception:
        GCP_PROJECT_ID = input("Your registered Earth Engine project ID: ").strip()
if not GCP_PROJECT_ID:
    raise ValueError("A registered Earth Engine project is required for ERA5-Land and NEX-GDDP.")
import ee
try:
    ee.Initialize(project=GCP_PROJECT_ID)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT_ID)
os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
print("Earth Engine connected.")


## 4. Check a daily rainfall series

In [ ]:
from datetime import date

from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

LAT, LON = -1.286, 36.817  # Nairobi, Kenya — swap in your own site

VARS = [
    ClimateVariable.precipitation,
]

df = ct.fetch_climate_data(
    source="chirps_v3_daily_rnl", allow_gee=False,
    location_coord=(LAT, LON),
    variables=VARS,
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    verbose=False,
)
df.head()

In [ ]:
# It's a normal DataFrame — plot, resample, export as usual.
df.set_index("date")["precipitation"].plot(
    figsize=(10, 3), title="Daily precipitation, Nairobi 2020 (CHIRPS v3)"
)

## LTE data and reproducible setup

Install the published toolkit above; do not clone or enter its source checkout. The example loads ERAgriculture LTE tables at a fixed revision and checks required columns.

The opening CHIRPS v3 demonstration uses precipitation only, for 2020, without Earth Engine. The LTE analysis below retains **ERA5 (`era_5`)**, which still requires Earth Engine. Changing to ERA5-Land or NASA POWER changes the climate input and requires separate scientific review.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

LTE_REVISION = "526c4ae01656d7889d6af9c6ce0f39eecf3a124f"
LTE_BASE = f"https://raw.githubusercontent.com/ERAgriculture/LTEs/{LTE_REVISION}/data"
for filename in ("lte_final.csv", "lte_arms.csv"):
    table = pd.read_csv(f"{LTE_BASE}/{filename}", low_memory=False)
    table.to_csv(filename, index=False)
lte_final = pd.read_csv("lte_final.csv", low_memory=False)
lte_arms = pd.read_csv("lte_arms.csv", low_memory=False)
for table, required in [(lte_arms, {"LTE.ID", "arm_id", "yield_t_ha", "yield_basis", "n_means"}),
                        (lte_final, {"LTE.ID", "obs_id", "ctrl_id", "yield_t_ha", "yield_c_t_ha"})]:
    assert required <= set(table), f"Missing columns: {required - set(table)}"
print("LTE revision:", LTE_REVISION, "Pairwise:", lte_final.shape, "Arms:", lte_arms.shape)
display(lte_arms[["LTE.ID", "Site.ID", "Product.Simple", "Variety", "yield_t_ha", "yield_basis"]].head())


In [ ]:
# ==============================================================================
# Step 5: Extract Sites with Crop Yield Data using lte_arms.csv
# ==============================================================================
import pandas as pd

# Use lte_arms to capture all unique sites with crop yield data (treatments + controls)
yield_mask = lte_arms["Out.SubInd"] == "Crop Yield"
yield_sites = sorted(
    lte_arms.loc[yield_mask, "Site.ID"].dropna().astype(str).unique()
)

print(
    f"{len(yield_sites)} sites available for yield trends in lte_arms.csv — "
    "pass any (or a distinctive part) to --site:"
)
for s in yield_sites:
    print(f"   {s}")

## Dynamic Yield & Rainfall stress Analysis at the selected site:

Main steps:

1. Set the site and climate source
2. Extract Climate Windows
3. Execute climate_toolkit for rainfall related hazards

In [ ]:
import io
import re
import subprocess
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ==============================================================================
#  CONFIGURATION: SITE QUERY & CLIMATE SOURCE
# ==============================================================================
SITE_QUERY = "Gourton Farm"  # Try "Gourton Farm", "Kouve", "Misamfu ARS", "Askov"
CLIMATE_SOURCE = "era_5"  # Try "era_5" or "nasa_power"
SEASON_DAYS = 150  # Estimated climate window duration
# ==============================================================================


def format_to_mm_dd(val):
    if pd.isna(val):
        raise ValueError("Missing climate-window start; choose a documented start date.")
    text = str(val).strip()
    if len(text) == 5 and text[2] == "-":
        return pd.Timestamp(f"2000-{text}").strftime("%m-%d")
    return pd.to_datetime(text, errors="raise").strftime("%m-%d")



def calculate_estimated_end(start_mmdd, days=150):
    return (pd.Timestamp(f"2000-{start_mmdd}") + pd.Timedelta(days=days - 1)).strftime("%m-%d")



def rainfall_record(lat, lon, crop_name, year, start_mmdd, source, days=150):
    """Use structured API output; preserve errors and raw rainfall-stress status."""
    import contextlib
    import io
    start = pd.Timestamp(f"{year}-{start_mmdd}")
    end = start + pd.Timedelta(days=days - 1)
    with contextlib.redirect_stdout(io.StringIO()):
        result = ct.evaluate_hazards(
            crop_name=crop_name.strip(), location_coord=(lat, lon),
            date_from=start.strftime("%Y-%m-%d"), date_to=end.strftime("%Y-%m-%d"),
            season_start=start.strftime("%Y-%m-%d"), season_end=end.strftime("%Y-%m-%d"),
            source=source)
    if "error" in result:
        raise RuntimeError(f"{crop_name}, {year}: {result['error']}")
    assessment = result.get("assessments", [result])
    if len(assessment) != 1:
        raise ValueError(f"Expected one explicit season; received {len(assessment)}")
    rain = assessment[0]["hazard_evaluation"]["precipitation"]
    raw = str(rain["status"])
    if raw == "no_stress":
        category = "No Stress"
    elif "moderate" in raw:
        category = "Moderate Stress"
    elif "severe" in raw or "extreme" in raw:
        category = "Severe Stress"
    else:
        raise ValueError(f"Unrecognized precipitation status: {raw}")
    if not pd.notna(rain["value_mm"]):
        raise ValueError("Missing seasonal precipitation")
    return {"Year": year, "Precipitation_mm": rain["value_mm"],
            "Hazard_Category": category, "Raw_Status": raw,
            "Season_Start": str(start.date()), "Season_End": str(end.date()),
            "Source": source}



def plot_rainfall_hazard_classification(
    hazard_df, site_name, crop_name, window_str
):
  """Plots seasonal rainfall amounts colored by hazard stress categories (mean line removed)."""
  if hazard_df.empty or 'Precipitation_mm' not in hazard_df.columns:
    print('⚠️ No precipitation hazard data available to plot.')
    return

  df = hazard_df.dropna(subset=['Precipitation_mm']).copy()
  if df.empty:
    print('⚠️ All precipitation values are missing.')
    return

  category_colors = {
      'No Stress': 'forestgreen',
      'Moderate Stress': 'darkorange',
      'Severe Stress': 'crimson',
      'Unknown': 'gray',
  }

  plt.figure(figsize=(13, 6))
  sns.set_style('whitegrid')

  status_handles = {}

  for idx, row in df.iterrows():
    yr = int(row['Year'])
    val = row['Precipitation_mm']
    cat = row['Hazard_Category']
    color = category_colors.get(cat, 'gray')

    plt.bar(
        yr,
        val,
        width=0.45,
        color=color,
        alpha=0.85,
        edgecolor='black',
        linewidth=0.8,
    )
    if cat not in status_handles:
      status_handles[cat] = mpatches.Patch(
          facecolor=color, edgecolor='black', label=cat, alpha=0.85
      )

  plt.title(
      f'{site_name} ({crop_name}): Annual Seasonal Rainfall Hazard'
      f' Assessment\n(Estimated Climate Window: {window_str})',
      fontsize=13,
      fontweight='bold',
      pad=15,
  )
  plt.xlabel('Season Year', fontsize=11, labelpad=8)
  plt.ylabel('Seasonal Total Precipitation (mm)', fontsize=11, labelpad=8)
  plt.xticks(df['Year'].astype(int), rotation=45)

  # Custom legend for Hazard Categories only (mean line removed)
  legend_patches = [
      status_handles[c] for c in category_colors.keys() if c in status_handles
  ]
  plt.legend(
      handles=legend_patches,
      title='Precipitation Hazard Status',
      bbox_to_anchor=(1.02, 1),
      loc='upper left',
      frameon=True,
  )

  plt.tight_layout()
  plot_file = f"{site_name.lower().replace(' ', '_')}_{crop_name.lower().replace(' ', '_')}_rainfall_hazard.png"
  plt.savefig(plot_file, dpi=300, bbox_inches='tight')
  plt.show()
  print(f"✓ Saved Rainfall Hazard plot: '{plot_file}'")


def run_seasonal_rainfall_hazard_analysis(
    site_name, lte_arms_df, source=CLIMATE_SOURCE, season_days=SEASON_DAYS
):
  """Main workflow: Extracts site/crop metadata and calculates annual seasonal precipitation

  hazard classification using climate_toolkit.calculate_hazards.hazards.
  """
  site_mask = (
      lte_arms_df['Site.ID'].str.contains(site_name, case=False, na=False)
  ) & (lte_arms_df['Out.SubInd'] == 'Crop Yield')

  site_data = lte_arms_df[site_mask].copy()

  if site_data.empty:
    print(f"❌ No crop yield records found for site query '{site_name}'.")
    return

  assert site_data['Site.ID'].nunique() == 1, 'Choose exactly one site.'
  matched_site = str(site_data['Site.ID'].iloc[0])
  crop_col = next(
      (
          c
          for c in site_data.columns
          if c.lower() in ['product.simple', 'product_simple', 'crop']
      ),
      None,
  )
  valid_crops = (
      site_data[crop_col].dropna().unique() if crop_col else [None]
  )

  print('==================================================')
  print(f" Seasonal Rainfall Hazard Workflow: '{matched_site}'")
  print('==================================================\n')

  for crop_val in valid_crops:
    if crop_val is not None and crop_col:
      crop_sub = site_data[site_data[crop_col] == crop_val].copy()
      crop_name = str(crop_val).strip()
    else:
      crop_sub = site_data.copy()
      crop_name = 'Maize'

    if crop_sub.empty:
      continue

    lat_col = next(
        (c for c in crop_sub.columns if c.lower() in ['latitude', 'lat']), None
    )
    lon_col = next(
        (
            c
            for c in crop_sub.columns
            if c.lower() in ['longitude', 'lon', 'long']
        ),
        None,
    )
    lat, lon = None, None
    if lat_col and lon_col:
      v_lats = crop_sub[lat_col].dropna()
      v_lons = crop_sub[lon_col].dropna()
      if not v_lats.empty and not v_lons.empty:
        lat = float(v_lats.iloc[0])
        lon = float(v_lons.iloc[0])

    cw_start_col = next(
        (c for c in crop_sub.columns if c.lower() == 'climate_window_start'),
        None,
    )
    raw_cw_start = (
        crop_sub[cw_start_col].dropna().iloc[0]
        if (cw_start_col and not crop_sub[cw_start_col].dropna().empty)
        else 'N/A'
    )

    rep_cw_start = format_to_mm_dd(raw_cw_start)
    est_cw_end = calculate_estimated_end(rep_cw_start, days=season_days)

    if 'M.Year' in crop_sub.columns:
      crop_sub['Year'] = pd.to_numeric(crop_sub['M.Year'], errors='coerce')
    elif 'm_year_first' in crop_sub.columns:
      crop_sub['Year'] = pd.to_numeric(
          crop_sub['m_year_first'], errors='coerce'
      )

    crop_sub = crop_sub.dropna(subset=['Year']).copy()
    crop_sub['Year'] = crop_sub['Year'].astype(int)

    start_year = int(crop_sub['Year'].min())
    end_year = int(crop_sub['Year'].max())

    print(
        '--------------------------------------------------------------------------'
    )
    print(f' CROP: {crop_name.upper()} ({matched_site})')
    print(
        '--------------------------------------------------------------------------'
    )
    print(f'✓ Coordinates: Latitude = {lat:.4f}, Longitude = {lon:.4f}')
    print(
        f'✓ Estimated Climate Window ({season_days} days): {rep_cw_start} to'
        f' {est_cw_end}'
    )
    print(f'✓ Experiment Period: {start_year} to {end_year}')
    print(f'✓ Climate Source: {source}\n')

    if lat is not None and lon is not None and rep_cw_start != 'N/A':

      hazard_records = []
      print(
          ' Calculating annual rainfall hazard via climate_toolkit for each'
          ' season year...'
      )

      for yr in range(start_year, end_year + 1):
        hazard_records.append(rainfall_record(lat, lon, crop_name, yr, rep_cw_start, source, season_days))

      hazard_df = pd.DataFrame(hazard_records)

      print(
          f'\n=== {matched_site} ({crop_name}): Annual Seasonal Rainfall Hazard'
          ' Table ==='
      )
      display(
          hazard_df[['Year', 'Season_Start', 'Season_End', 'Precipitation_mm', 'Hazard_Category', 'Raw_Status']].round(2)
      )

      plot_rainfall_hazard_classification(
          hazard_df, matched_site, crop_name, f'{rep_cw_start} to {est_cw_end}'
      )

    else:
      print('⚠️ Coordinates or climate window unavailable.')


# ==============================================================================
# EXECUTE RAINFALL HAZARD ANALYSIS
# ==============================================================================
run_seasonal_rainfall_hazard_analysis(SITE_QUERY, lte_arms)

## Exploratory treatment–control yield comparisons

Use normalized `yield_t_ha` and `yield_c_t_ha` columns. Retain the source treatment descriptions and raw precipitation status in verification tables.

**Method corrections:** each window contains exactly 150 days starting in the stated season year, including windows crossing New Year. This corrects the earlier inclusive-day offset. Historical figures must therefore be regenerated. The 150-day duration remains a teaching assumption, not a validated crop calendar.

“Stress” below pools low- and high-rainfall threshold statuses; it must **not** be labelled drought. Tables are descriptive averages across reported comparisons, not evidence of causal adaptation benefits. Check crop, season, units, replication, repeated controls and uncertainty before scientific interpretation. Thresholds measure rainfall stress, not all climate hazards.

The previous duplicate analysis cell is removed; the following cell is the single implementation.


In [ ]:
import io
import re
import subprocess
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ==============================================================================
#  CONFIGURATION: TARGET SITE & CLIMATE SOURCE
# ==============================================================================
CSV_FILE_PATH = "lte_final.csv"
SITE_QUERY = "Gourton Farm"  # Works for any site: "Gourton Farm", "Misamfu ARS", "Kouve", etc.
CLIMATE_SOURCE = "era_5"
SEASON_DAYS = 150
# ==============================================================================











def fetch_climate_hazards_cached(lat, lon, crop_name, start_year, end_year, cw_start, cw_end, source, cache):
    records = []
    for yr in range(start_year, end_year + 1):
        key = (lat, lon, crop_name, yr, cw_start, SEASON_DAYS, source)
        if key not in cache:
            cache[key] = rainfall_record(lat, lon, crop_name, yr, cw_start, source, SEASON_DAYS)
        records.append(cache[key])
    return pd.DataFrame(records)



def assign_display_category(row, subpr_count_in_theme):
    # Keep source treatment labels; do not infer tillage classes from substrings.
    subpr = str(row.get("SubPrName", "")).strip()
    if subpr_count_in_theme > 1:
        return subpr
    return f"{subpr}: {row.get('T.Descrip', '')} vs {row.get('C.Descrip', '')}"



def run_multi_theme_sequential_analysis(
    csv_path=CSV_FILE_PATH, site_query=SITE_QUERY
):
  """Executes sequential theme-by-theme analysis for any given site in lte_final.csv,

  dynamically resolving SubPrName or T.Descrip/C.Descrip categories.
  """
  df = pd.read_csv(csv_path, low_memory=False)

  # 1. Base Filter for Site, Crop Yield, and Non-aggregated Single Years
  mask_base = (
      (df['Site.ID'].str.contains(site_query, case=False, na=False))
      & (df['Out.SubInd'] == 'Crop Yield')
      & (
          df['m_year_is_aggregate'].isna()
          | (df['m_year_is_aggregate'] == 0)
          | df['m_year_is_aggregate'].eq(False)
      )
  )

  site_df = df[mask_base].copy()

  if site_df.empty:
    print(f"❌ No valid yield records found for site '{site_query}'.")
    return

  assert site_df['Site.ID'].nunique() == 1, 'Choose exactly one site.'
  assert site_df['Product.Simple'].dropna().nunique() == 1, 'Filter to one crop before comparing yields.'
  matched_site = str(site_df['Site.ID'].dropna().iloc[0])
  lat = float(site_df['Latitude'].dropna().iloc[0])
  lon = float(site_df['Longitude'].dropna().iloc[0])

  cw_start_col = next(
      (c for c in site_df.columns if c.lower() == 'climate_window_start'),
      None,
  )
  raw_cw_start = (
      site_df[cw_start_col].dropna().iloc[0]
      if (cw_start_col and not site_df[cw_start_col].dropna().empty)
      else '09-01'
  )
  rep_cw_start = format_to_mm_dd(raw_cw_start)
  est_cw_end = calculate_estimated_end(rep_cw_start, days=SEASON_DAYS)

  crop_name = (
      str(site_df['Product.Simple'].dropna().iloc[0])
      if (
          'Product.Simple' in site_df.columns
          and not site_df['Product.Simple'].dropna().empty
      )
      else 'Maize'
  )

  # Clean Numeric Columns
  site_df['Year'] = pd.to_numeric(
      site_df.get('M.Year', site_df.get('m_year_first')), errors='coerce'
  )
  site_df['MeanT'] = pd.to_numeric(site_df['yield_t_ha'], errors='coerce')
  site_df['MeanC'] = pd.to_numeric(site_df['yield_c_t_ha'], errors='coerce')

  site_df = site_df.dropna(subset=['Year', 'MeanT', 'MeanC']).copy()
  site_df['Year'] = site_df['Year'].astype(int)

  # Identify All Unique Themes at this Site
  themes_list = [
      t for t in site_df['Theme'].dropna().unique() if str(t).strip() != ''
  ]
  total_themes = len(themes_list)

  print(
      '=========================================================================================='
  )
  print(f" MULTI-THEME SEQUENTIAL ANALYSIS PIPELINE — SITE: '{matched_site}'")
  print(f'   Total Themes Identified: {total_themes}')
  for idx, t in enumerate(themes_list, 1):
    print(f'   • Theme {idx}: {t}')
  print(
      f'   Crop: {crop_name} | Coordinates: ({lat:.4f}, {lon:.4f}) | Climate'
      f' Window: {rep_cw_start} to {est_cw_end}'
  )
  print(
      '==========================================================================================\n'
  )

  climate_cache = {}

  # 2. Iterate Theme by Theme
  for theme_idx, theme_name in enumerate(themes_list, 1):
    theme_sub = site_df[site_df['Theme'] == theme_name].copy()

    # Determine unique SubPrName count for this theme
    subpr_count = theme_sub['SubPrName'].nunique()

    # Assign dynamic Display_Category
    theme_sub['Display_Category'] = theme_sub.apply(
        lambda r: assign_display_category(r, subpr_count), axis=1
    )

    print(
        '=========================================================================================='
    )
    print(f" THEME {theme_idx} OF {total_themes}: '{theme_name}'")
    print(
        f'   Site: {matched_site} | Records: {len(theme_sub)} |'
        f" Categories: {list(theme_sub['Display_Category'].unique())}"
    )
    print(
        '==========================================================================================\n'
    )

    # Verification Table 1: Annual Reported Averages by PrName & Display_Category
    yearly_verify_df = (
        theme_sub.groupby(['PrName', 'SubPrName', 'Display_Category', 'Year'])
        .agg(
            MeanT_avg=('MeanT', 'mean'),
            MeanC_avg=('MeanC', 'mean'),
            N_Obs=('MeanT', 'count'),
            T_Descrip_Included=(
                'T.Descrip',
                lambda x: ' | '.join(sorted(list(set(x.dropna())))),
            ),
            C_Descrip_Included=(
                'C.Descrip',
                lambda x: ' | '.join(sorted(list(set(x.dropna())))),
            ),
        )
        .reset_index()
    )

    yearly_verify_df['ΔY_avg (t/ha)'] = (
        yearly_verify_df['MeanT_avg'] - yearly_verify_df['MeanC_avg']
    )

    print(
        f' VERIFICATION TABLE 1 (Theme {theme_idx}): Annual Reported Averages'
        ' (MeanT & MeanC)'
    )
    cols_verify = [
        'PrName',
        'SubPrName',
        'Display_Category',
        'Year',
        'N_Obs',
        'MeanT_avg',
        'MeanC_avg',
        'ΔY_avg (t/ha)',
        'T_Descrip_Included',
        'C_Descrip_Included',
    ]
    display(
        yearly_verify_df[cols_verify]
        .sort_values(['PrName', 'Display_Category', 'Year'])
        .round(2)
    )

    # Fetch Climate Hazards for Theme Years
    start_year = int(yearly_verify_df['Year'].min())
    end_year = int(yearly_verify_df['Year'].max())

    hazard_df = fetch_climate_hazards_cached(
        lat,
        lon,
        crop_name,
        start_year,
        end_year,
        rep_cw_start,
        est_cw_end,
        CLIMATE_SOURCE,
        climate_cache,
    )

    # Verification Table 2: Annual Yields Mapped with Climate Hazards
    merged_annual = pd.merge(
        yearly_verify_df, hazard_df, on='Year', how='left'
    )

    print(
        f'\n VERIFICATION TABLE 2 (Theme {theme_idx}): Annual Yields Mapped'
        ' with Climate Hazards'
    )
    cols_hazard_display = [
        'PrName',
        'SubPrName',
        'Display_Category',
        'Year',
        'Precipitation_mm',
        'Hazard_Category',
        'Raw_Status',
        'MeanT_avg',
        'MeanC_avg',
        'ΔY_avg (t/ha)',
    ]
    display(merged_annual[cols_hazard_display].round(2))

    # Summary Table 3: Yield & Delta Yield Summary (Normal vs Stress)
    stress_cats = ['Moderate Stress', 'Severe Stress']
    normal_df = merged_annual[merged_annual['Hazard_Category'] == 'No Stress']
    stress_df = merged_annual[merged_annual['Hazard_Category'].isin(stress_cats)]

    adaptation_summary = []
    unique_pr_cats = merged_annual[
        ['PrName', 'SubPrName', 'Display_Category']
    ].drop_duplicates()

    for _, row_tax in unique_pr_cats.iterrows():
      p_name, sp_name, cat_name = (
          row_tax['PrName'],
          row_tax['SubPrName'],
          row_tax['Display_Category'],
      )

      sub_norm = normal_df[
          (normal_df['PrName'] == p_name)
          & (normal_df['Display_Category'] == cat_name)
      ]
      sub_stress = stress_df[
          (stress_df['PrName'] == p_name)
          & (stress_df['Display_Category'] == cat_name)
      ]

      y_norm_t = sub_norm['MeanT_avg'].mean() if not sub_norm.empty else np.nan
      y_norm_c = sub_norm['MeanC_avg'].mean() if not sub_norm.empty else np.nan
      delta_norm = y_norm_t - y_norm_c if pd.notna(y_norm_t) else np.nan

      y_stress_t = (
          sub_stress['MeanT_avg'].mean() if not sub_stress.empty else np.nan
      )
      y_stress_c = (
          sub_stress['MeanC_avg'].mean() if not sub_stress.empty else np.nan
      )
      delta_stress = (
          y_stress_t - y_stress_c if pd.notna(y_stress_t) else np.nan
      )

      adaptation_summary.append({
          'Theme': theme_name,
          'PrName': p_name,
          'SubPrName': sp_name,
          'Practice Category': cat_name,
          'MeanT Normal (t/ha)': (
              round(y_norm_t, 2) if pd.notna(y_norm_t) else None
          ),
          'MeanC Normal (t/ha)': (
              round(y_norm_c, 2) if pd.notna(y_norm_c) else None
          ),
          'ΔY Normal (t/ha)': (
              round(delta_norm, 2) if pd.notna(delta_norm) else None
          ),
          'MeanT Stress (t/ha)': (
              round(y_stress_t, 2) if pd.notna(y_stress_t) else None
          ),
          'MeanC Stress (t/ha)': (
              round(y_stress_c, 2) if pd.notna(y_stress_c) else None
          ),
          'ΔY Stress (t/ha)': (
              round(delta_stress, 2) if pd.notna(delta_stress) else None
          ),
      })

    summary_df = pd.DataFrame(adaptation_summary)

    print(
        f'\n SUMMARY TABLE 3 (Theme {theme_idx}): Yield & Delta Yield Summary'
        ' (Normal vs Stress)'
    )
    display(summary_df)

    # Plotting Trends for this Theme
    fig, ax1 = plt.subplots(figsize=(14, 6))
    sns.set_style('white')

    category_colors = {
        'No Stress': 'forestgreen',
        'Moderate Stress': 'darkorange',
        'Severe Stress': 'crimson',
    }

    years = hazard_df['Year'].values
    precip = hazard_df['Precipitation_mm'].values
    cats = hazard_df['Hazard_Category'].values

    for idx, yr in enumerate(years):
      ax1.bar(
          yr,
          precip[idx],
          width=0.4,
          color=category_colors.get(cats[idx], 'gray'),
          alpha=0.45,
          edgecolor='black',
      )

    ax1.set_xlabel('Season Year', fontsize=11, labelpad=8)
    ax1.set_ylabel(
        'Seasonal Precipitation (mm)',
        fontsize=11,
        fontweight='bold',
        color='black',
    )
    ax1.set_ylim(0, max(precip) * 1.15)
    ax1.set_xticks(years)

    ax2 = ax1.twinx()
    unique_cats = merged_annual['Display_Category'].unique()
    palette = sns.color_palette('tab10', n_colors=len(unique_cats))

    for idx, cat_name in enumerate(unique_cats):
      sub_data = merged_annual[merged_annual['Display_Category'] == cat_name]
      if not sub_data.empty:
        ax2.plot(
            sub_data['Year'],
            sub_data['ΔY_avg (t/ha)'],
            label=f'ΔY: {cat_name}',
            color=palette[idx % len(palette)],
            linewidth=2.5,
            marker='o',
        )

    ax2.axhline(0, color='black', linestyle='--', linewidth=1.2, label='Control Baseline (ΔY=0)')
    ax2.set_ylabel(
        'Yield Advantage ΔY (t/ha)',
        fontsize=11,
        fontweight='bold',
        color='darkblue',
    )
    ax2.grid(False)

    plt.title(
        f'{matched_site} ({crop_name}) — Theme {theme_idx}: {theme_name}\nYield'
        ' Advantage (ΔY = MeanT_avg - MeanC_avg) vs Climate Hazards',
        fontsize=12,
        fontweight='bold',
        pad=15,
    )

    hazard_patches = [
        mpatches.Patch(
            facecolor=c, alpha=0.45, label=label, edgecolor='black', linewidth=0.8
        )
        for label, c in category_colors.items()
        if label in cats
    ]
    ax1.legend(
        handles=hazard_patches,
        title='Rainfall Hazard',
        loc='upper left',
        bbox_to_anchor=(1.12, 1.0),
    )
    ax2.legend(
        title='Practice Categories ΔY',
        loc='upper left',
        bbox_to_anchor=(1.12, 0.55),
    )

    plt.tight_layout()
    slug_site = matched_site.lower().replace(' ', '_')
    plot_filename = f'{slug_site}_theme_{theme_idx}_adaptation.png'
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved plot for Theme {theme_idx}: '{plot_filename}'\n\n")


# Execute Multi-Theme Pipeline
run_multi_theme_sequential_analysis(CSV_FILE_PATH, SITE_QUERY)